# Visite guidée du projet « Quel opérateur chez moi ? »

**À quoi sert ce notebook :** présenter le projet à ta binôme, étape par étape, dans l'ordre où les données circulent.
Chaque étape a le même format :
- 📂 **Fichiers** : ce qu'on ouvre
- 🗣️ **Quoi dire** : les phrases clés (à dire avec tes mots)
- ▶️ **Montrer** : une cellule à lancer (aucune n'a besoin d'Internet)

**Durée :** ~45 min. **Avant de commencer :** ouvrir ce notebook depuis la racine du projet, avec le Python de `.venv` (kernel `.venv`).

In [1]:
# Vérification : on est bien à la racine du projet (sinon « import config » échoue)
import pathlib
print(pathlib.Path.cwd())
assert pathlib.Path("config.py").exists(), "Ouvre le notebook depuis la racine du projet"

C:\Users\Utilisateur\Desktop\best mobile operator


---
## Étape 0 — Le projet en 1 minute

🗣️ **Quoi dire**
- **Le problème :** les 4 opérateurs disent tous « meilleur réseau ». Les vraies données existent (ANFR, Arcep) mais sont éparpillées et illisibles.
- **Pour qui :** quelqu'un qui déménage ou change de forfait.
- **Ce que fait l'appli :** on tape une adresse → score /100 pour Orange, SFR, Bouygues, Free → verdict « chez toi, prends X » + carte des antennes + conseil « marque moins chère sur le même réseau » (Sosh, RED by SFR, B&You).
- **Le lien avec le cours :** on suit le plan du cours, **Linux → Git → Docker**, jusqu'au déploiement (85 % des projets data échouent faute de mise en production).

▶️ **Montrer :** lancer l'appli dans Git Bash et taper « 8 rue du Port, Nanterre ».
```bash
bash run.sh        # la page s'ouvre sur http://localhost:8501
```

---
## Étape 1 — L'architecture : séparer « préparer » et « répondre »

```
INTERNET (open data)           PIPELINE (en fond, toutes les 24 h)                 APPLI (à chaque visite)

ANFR (mensuel) ───────┐
Arcep mesures (annuel) ┼─► extraction.sh ► data_raw/ ─► clean.py ──────► data/ ─► app.py + score.py ─► navigateur
Arcep couverture (trim.)┘  (si nouveau)  (supprimé)  + couverture.py  (27 Mo,   + API Adresse (IGN)
                                                                       dans Git)
```

🗣️ **Quoi dire**
- **Deux parties :** le **pipeline** (lent, rare : télécharger et nettoyer) et l'**appli** (rapide : lire `data/` et calculer autour de l'adresse).
- **Un dossier par rôle**, comme l'appli exemple du prof : `pipeline/`, `webapp/`, `config.py`, `log/`, `archived/`, `bin/`. Chaque dossier a son README.
- **Les données préparées sont dans Git** (~27 Mo) : un `git clone` démarre tout de suite, sans télécharger 1,5 Go.

In [2]:
# L'arborescence du projet (sans les dossiers techniques)
for p in sorted(pathlib.Path(".").rglob("*")):
    if any(x in p.parts for x in [".git", ".venv", "__pycache__", "log", "archived", "data_raw", ".pytest_cache", ".ipynb_checkpoints"]):
        continue
    if p.is_file():
        print(f"{str(p):45} {p.stat().st_size / 1e6:7.2f} Mo")

.dockerignore                                    0.00 Mo
.gitattributes                                   0.00 Mo
.github\workflows\ci.yml                         0.00 Mo
.gitignore                                       0.00 Mo
01_exploration.ipynb                             0.54 Mo
02_visite_guidee.ipynb                           0.05 Mo
bin\trouver_python.sh                            0.00 Mo
CLAUDE.md                                        0.02 Mo
config.py                                        0.00 Mo
data\couverture_4g.npz                          21.41 Mo
data\derniere_version.json                       0.00 Mo
data\mesures.csv.gz                              1.21 Mo
data\sites.csv.gz                                4.89 Mo
DICTIONNAIRE_DONNEES.md                          0.02 Mo
Dockerfile                                       0.00 Mo
install.sh                                       0.00 Mo
pipeline\__init__.py                             0.00 Mo
pipeline\archive.py            

---
## Étape 2 — Les données : 3 sources ouvertes + 1 API

| | **ANFR** | **Arcep – mesures** | **Arcep – couverture** | **API Adresse (IGN)** |
|---|---|---|---|---|
| Question | Où sont les antennes ? | Ça marche **en vrai** ? | Que dit la carte officielle ? | Adresse → GPS |
| Type | Déclarations officielles | **Mesures** de terrain | **Simulations** des opérateurs | En direct |
| Mise à jour | Mensuelle (nov. 2026) | Annuelle (campagne 2026) | Trimestrielle (T2 2026) | — |
| Brut → gardé | ~60 Mo → 4,9 Mo | 93 Mo → 1,4 Mo | 3 Go → 21 Mo | rien |
| Poids dans le score | 20 % | 50 % | 30 % | — |

🗣️ **Quoi dire**
- Les 3 sources **se complètent** : une simulation (couverture), des mesures réelles (Arcep), la réalité physique (antennes).
- Tout vient de **data.gouv.fr / data.arcep.fr / Géoplateforme IGN**, gratuit, sans clé.
- 📂 Le détail de chaque colonne est dans `DICTIONNAIRE_DONNEES.md`, l'exploration de départ dans `01_exploration.ipynb`.

---
## Étape 3 — `pipeline/extraction.sh` : télécharger seulement le nouveau (en bash)

📂 `pipeline/extraction.sh`, `pipeline/grille.py`, `config.py`, `data/derniere_version.json`

🗣️ **Quoi dire**
- Pour chaque source : **dernière version en ligne → comparer avec le carnet → si identique, stop → sinon télécharger, faire le ménage, noter dans le carnet EN DERNIER** (si ça plante au milieu, le passage suivant recommence).
- **Le carnet** = `data/derniere_version.json`. Grâce à lui, un passage sans nouveauté prend **5 secondes**.
- Téléchargement **par morceaux de 1 Mo** (jamais 93 Mo en mémoire) ; couverture traitée **un opérateur à la fois** (~1 Go max sur le disque).
- Les années Arcep sont **parcourues** : l'Arcep a déjà déplacé ses dossiers (erreur 404 qu'on a corrigée).
- Toutes les adresses et chemins sont dans **`config.py`**, rien en dur (le script bash les y lit aussi).
- **L'extraction est en bash avec `curl`**, comme l'appli exemple du prof. Python n'intervient que pour ce que bash ne sait pas faire : ouvrir les `.7z` de couverture et calculer la grille (`grille.py`).
- (`collect.py` fait la même chose en Python : il reste pour le notebook d'exploration.)

In [3]:
# Le carnet des versions : ce que collect.py a déjà téléchargé
import json
print(json.dumps(json.load(open("data/derniere_version.json", encoding="utf-8")), indent=2, ensure_ascii=False))

{
  "anfr": "https://static.data.gouv.fr/resources/donnees-sur-les-installations-radioelectriques-de-plus-de-5-watts-1/20261106-083007/20261031-export-etalab-data.zip https://static.data.gouv.fr/resources/donnees-sur-les-installations-radioelectriques-de-plus-de-5-watts-1/20261106-083015/20261031-export-etalab-ref.zip",
  "anfr_mois": "novembre 2026",
  "arcep": "https://data.arcep.fr/mobile/mesures_qualite_arcep/2026/Metropole/2026_QoS_Metropole_data_habitations.csv",
  "couverture": [
    "https://data.arcep.fr/mobile/couvertures_theoriques/last/Metropole/00_Metropole/2026_T2_couv_Metropole_BOUY_4G_data.gpkg.7z",
    "https://data.arcep.fr/mobile/couvertures_theoriques/last/Metropole/00_Metropole/2026_T2_couv_Metropole_FREE_4G_data.gpkg.7z",
    "https://data.arcep.fr/mobile/couvertures_theoriques/last/Metropole/00_Metropole/2026_T2_couv_Metropole_OF_4G_data.gpkg.7z",
    "https://data.arcep.fr/mobile/couvertures_theoriques/last/Metropole/00_Metropole/2026_T2_couv_Metropole_SFR0_4G_d

---
## Étape 4 — `pipeline/clean.py` : du brut aux fichiers propres

📂 `pipeline/clean.py`, `pipeline/archive.py`

🗣️ **Quoi dire**
- **ANFR** : 3 tables techniques reliées par la clé `STA_NM_ANFR` → **une table simple** : quelle antenne, quel opérateur, quelle techno (4G = `LTE 800`…, 5G = `5G NR 3500`…), où, depuis quand.
  - On garde les 4 opérateurs (codes 6, 23, 137, 240), la 4G/5G et la métropole (codes INSEE 97/98 exclus).
  - Positions converties : degrés/minutes/secondes → degrés décimaux (`48° 51' 24"` → `48,8567`).
- **Arcep** : on garde 6 colonnes (~95 Mo → 1,2 Mo) et on crée `reussi` (vrai/faux). Le format a changé en 2026 (plus de colonne `Result`) : on utilise « page web chargée en moins de 10 s » et « vidéo de qualité correcte » — c'était exactement la définition du succès en 2025.
- **Écriture sans risque** : fichier temporaire puis `os.replace` → l'appli ne lit jamais un fichier à moitié écrit. L'ancienne version part dans `archived/` (on peut revenir en arrière).
- Ensuite **le brut est supprimé**.

In [4]:
import pandas as pd
from config import SITES, MESURES
sites = pd.read_csv(SITES, dtype={"site_id": str, "code_insee": str}, parse_dates=["date_service"])
mesures = pd.read_csv(MESURES)

print("sites.csv.gz :", len(sites), "lignes,", sites["site_id"].nunique(), "sites,", sites["code_insee"].nunique(), "communes")
display(sites.head())
print("Sites par opérateur et techno :")
display(sites.drop_duplicates(["site_id", "techno"]).pivot_table(index="operateur", columns="techno",
        values="site_id", aggfunc="count"))

sites.csv.gz : 641292 lignes, 134435 sites, 21828 communes


,site_id,operateur,techno,frequence,date_service,latitude,longitude,code_insee
0,0012750479,Bouygues,4G,700,2023-08-09,46.138333,4.93,01001
1,0012750479,Bouygues,4G,800,2021-09-02,46.138333,4.93,01001
2,0010990357,Free,4G,700,2021-09-30,46.138333,4.93,01001
3,0010990357,Free,4G,800,NaT,46.138333,4.93,01001
4,0012290398,Orange,4G,700,2023-05-04,46.138333,4.93,01001


Sites par opérateur et techno :


techno,4G,5G
operateur,,
Bouygues,34137,22730
Free,33669,26455
Orange,33723,21131
SFR,32888,20194


In [5]:
print("mesures.csv.gz :", len(mesures), "tests Arcep")
display(mesures.head())
print("Moyennes nationales par opérateur :")
display(mesures.groupby("operateur").agg(nb_tests=("reussi", "size"), taux_reussite=("reussi", "mean"),
                                        debit_median=("debit", "median")).round(2))

mesures.csv.gz : 338823 tests Arcep


,operateur,latitude,longitude,protocole,reussi,debit
0,Bouygues,46.13889,5.90526,WEB,1.0,NaN
1,Free,46.13889,5.90526,WEB,1.0,NaN
2,SFR,46.13889,5.90526,WEB,1.0,NaN
3,Orange,46.13889,5.90526,WEB,1.0,NaN
4,Bouygues,46.13889,5.90526,WEB,1.0,NaN


Moyennes nationales par opérateur :


,nb_tests,taux_reussite,debit_median
operateur,,,
Bouygues,84710,0.93,155.30
Free,84694,0.93,103.88
Orange,84708,0.95,253.59
SFR,84711,0.93,140.37


---
## Étape 5 — `pipeline/couverture.py` : 3 Go de polygones → une grille de 21 Mo

🗣️ **Quoi dire**
- Les cartes Arcep sont des **formes** (52 millions de points par opérateur) : trop lent de chercher « dans quel polygone est l'adresse ? » à chaque visite.
- On découpe la France en **cases de 100 m** (122 millions de cases) et on note **à l'avance** le niveau dans chaque case — comme une photo : on remplace un dessin par des pixels.
- **Astuce :** un niveau (0 à 3) tient sur 2 bits → les **4 opérateurs dans 1 octet**. Lecture d'une adresse : **0,03 ms**.
- C'est une **approximation** : 79 points sur 80 identiques au calcul exact.
- Niveaux : **TBC** = marche aussi à l'intérieur ; **BC** = dehors, parfois dedans ; **CL** = dehors seulement. Ce sont des **simulations des opérateurs** → seulement 30 % du score.

In [6]:
import numpy as np
from config import GRILLE
from pipeline.couverture import couverture_autour, VERS_L93, X0, Y1, PAS

with np.load(GRILLE) as f:
    grille = f["grille"]
print("Grille :", grille.shape, "=", grille.size, "cases de 100 m ;", round(GRILLE.stat().st_size / 1e6), "Mo sur le disque")

lat, lon = 48.893461, 2.178892                       # 8 rue du Port, Nanterre
x, y = VERS_L93.transform(lon, lat)
case = int(grille[int((Y1 - y) // PAS), int((x - X0) // PAS)])
print(f"Nanterre -> octet {case} = {case:08b} en binaire (SFR | Orange | Free | Bouygues, 2 bits chacun)")
couverture_autour(lat, lon)

Grille : (10650, 11440) = 121836000 cases de 100 m ; 21 Mo sur le disque
Nanterre -> octet 255 = 11111111 en binaire (SFR | Orange | Free | Bouygues, 2 bits chacun)


{'Bouygues': 'Très bonne couverture',
 'Free': 'Très bonne couverture',
 'Orange': 'Très bonne couverture',
 'SFR': 'Très bonne couverture'}

---
## Étape 6 — Le calcul du verdict : `webapp/adresse.py`, `antennes_proches.py`, `score.py`

🗣️ **Quoi dire** (c'est **LA** question du board)
1. **Adresse → GPS** avec l'API Adresse de l'IGN (`adresse.py`).
2. **Autour de l'adresse** (`antennes_proches.py`) : antennes à moins d'**1 km**, tests Arcep à moins de **2 km** (distance de haversine = à vol d'oiseau sur la Terre).
3. **Lissage** : s'il y a peu de tests près de chez toi, on se rapproche de la moyenne nationale (`K = 20`, « comme 20 tests en plus »). Sinon un seul test raté pouvait faire perdre un opérateur.
4. **5 notes de 0 à 1**, puis une moyenne pondérée (`score.py`) :

| Critère | Poids | Source |
|---|---|---|
| Couverture 4G | 30 % | Arcep couverture |
| Tests réussis | 30 % | Arcep mesures |
| Débit | 20 % | Arcep mesures |
| 5G rapide (3,5 GHz) | 10 % | ANFR |
| Proximité de l'antenne | 10 % | ANFR |

⚠️ **Limite connue** : là où il n'y a aucun test Arcep (ex. Pontivy), les 4 opérateurs reçoivent la **même** moyenne nationale → seuls la 5G et la distance les départagent.

In [7]:
# Le calcul complet pour Nanterre (l'API Adresse est remplacée par la position connue : pas besoin d'Internet)
import webapp.score as score
score.geocoder = lambda adresse: {"adresse": "8 Rue du Port 92000 Nanterre", "latitude": 48.893461,
                                  "longitude": 2.178892, "code_insee": "92050", "confiance": 1}
lieu, tableau = score.score_adresse("8 rue du Port, Nanterre", sites, mesures)
print("Poids :", score.POIDS)
display(tableau[["nb_tests", "taux_ajuste", "debit_ajuste", "couverture_4g", "nb_sites_5g_3500", "plus_proche_km",
                 "note_couverture", "note_reussite", "note_debit", "note_5g", "note_proximite", "score"]])
print("Verdict : chez toi, prends", tableau.index[0])

Poids : {'note_couverture': 0.3, 'note_reussite': 0.3, 'note_debit': 0.2, 'note_5g': 0.1, 'note_proximite': 0.1}


,nb_tests,taux_ajuste,debit_ajuste,couverture_4g,nb_sites_5g_3500,plus_proche_km,note_couverture,note_reussite,note_debit,note_5g,note_proximite,score
operateur,,,,,,,,,,,,
Free,32,98.0,149.0,Très bonne couverture,7,0.30,1.0,0.98,1.000000,1.000000,1.000000,0.99
SFR,32,98.0,137.0,Très bonne couverture,6,0.36,1.0,0.98,0.919463,0.857143,0.833333,0.95
Bouygues,32,98.0,135.0,Très bonne couverture,6,0.36,1.0,0.98,0.906040,0.857143,0.833333,0.94
Orange,32,90.0,131.0,Très bonne couverture,4,0.56,1.0,0.90,0.879195,0.571429,0.535714,0.86


Verdict : chez toi, prends Free


---
## Étape 7 — L'appli web : `webapp/app.py` + `templates/` + `static/`

🗣️ **Quoi dire**
- **Flask** sert la page et une petite **API JSON** : `/` (la page), `/api/adresses` (suggestions pendant la frappe), `/api/score` (verdict + classement + antennes), `/sante` (supervision).
- La page (`index.html` + `style.css` + `app.js`) appelle l'API et dessine les cartes des opérateurs et la **carte Leaflet** (fond Plan IGN).
- Les données sont lues **une fois**, puis relues seulement quand le pipeline les a mises à jour (pas besoin de redémarrer).
- Servie par **waitress** : un vrai serveur (celui de Flask n'est fait que pour développer), sous Windows, Mac et Linux.
- Pourquoi Flask + HTML et pas Streamlit (notre 1re version) : un vrai design de site, et l'API séparée de la page.

In [8]:
# On interroge l'appli comme le ferait le navigateur, sans lancer de serveur
# (le géocodage reste remplacé par la position de Nanterre : cellule précédente)
from webapp.app import app
client = app.test_client()
print("/sante ->", client.get("/sante").json)
reponse = client.get("/api/score?adresse=8 rue du Port, Nanterre").json
print("Verdict :", reponse["gagnant"], "| conseil :", reponse["moins_cher"], "|", len(reponse["antennes"]), "antennes à moins d'1 km")
[(op["nom"], op["score"]) for op in reponse["operateurs"]]

/sante -> {'fichiers': {'couverture_4g.npz': True, 'mesures.csv.gz': True, 'sites.csv.gz': True}, 'statut': 'ok'}


127.0.0.1 GET /api/score?adresse=8 rue du Port, Nanterre 200
Verdict : Free | conseil : None | 24 antennes à moins d'1 km


[('Free', 99), ('SFR', 95), ('Bouygues', 94), ('Orange', 86)]

---
## Étape 8 — Lancer partout : `install.sh`, `run.sh`, `bin/trouver_python.sh`, `config.py`

🗣️ **Quoi dire**
- **`install.sh`** (une fois) : trouve Python, crée l'environnement `.venv`, installe `requirements.txt` (versions exactes), crée `log/` et `archived/`.
- **`run.sh`** : lance le **pipeline en boucle en arrière-plan** (tout de suite, puis toutes les 24 h, un log par passage dans `log/`) et l'**appli au premier plan** (log dans `log/webapp.log`). Ctrl+C arrête tout.
- **`bin/trouver_python.sh`** : trouve un Python 3.12 à 3.14 quel que soit son nom (`python3`, `python`, `py`…).
- **Testé sur Windows (Git Bash), Mac (bash 3.2), Ubuntu serveur et Docker.**
- Pièges rencontrés : `python` introuvable sur Ubuntu, `pip install` refusé hors `.venv` (Ubuntu récent, Mac), fins de ligne Windows qui cassent bash sous Linux.

In [9]:
print(open("run.sh", encoding="utf-8").read())

#!/bin/bash
# run.sh : l'appli tout de suite (les données sont dans le dépôt) + le pipeline en boucle en arrière-plan
# Marche sous Windows (Git Bash), Mac, Linux et dans Docker.

cd "$(dirname "$0")" || exit 1   # on travaille toujours depuis le dossier du projet
export PYTHONIOENCODING=utf-8
export PYTHONUNBUFFERED=1        # les messages arrivent tout de suite dans les logs
INTERVALLE=86400                 # temps entre 2 passages, en secondes (86400 = 24 h)
mkdir -p log archived

# Le Python de .venv (créé par install.sh), sinon celui de l'ordinateur (ex. dans Docker)
source bin/trouver_python.sh
if [ -z "$PYTHON" ]; then
    echo "Python 3.12 à 3.14 introuvable : lance d'abord « bash install.sh »"
    exit 1
fi

# Quand le script s'arrête (Ctrl+C), on arrête aussi la boucle
trap 'kill 0' EXIT

# --- Le pipeline, rangé dans une fonction ---
# extraction.sh (bash + curl) ne télécharge que si une source a publié une nouvelle version,
# clean.py ne traite que ce qui vient d'être téléc

---
## Étape 9 — Git : versionner et travailler à deux

📂 `.gitignore`, `.gitattributes`, l'historique des commits

🗣️ **Quoi dire**
- **`.gitignore`** : on ne versionne pas `.venv/`, le brut, les logs, les archives. Mais **`data/` est versionné** (données préparées, 27 Mo).
- **`.gitattributes`** : `*.sh` toujours en fin de ligne Linux (LF), sinon `$'\r': command not found` dans Docker. Les `.gz` / `.npz` sont binaires.
- **Gzip sans date** (`mtime=0`) : mêmes données = même fichier → Git ne voit pas de faux changement.
- Dépôt GitHub `raoufRahmani/Best-mobile-operator`, branche `master`.

In [10]:
import subprocess
print(subprocess.run(["git", "log", "--oneline", "-15"], capture_output=True, text=True, encoding="utf-8").stdout)

336528e Mesures Arcep 2026 : clean.py lit l'ancien et le nouveau format (sans colonne Result : page web < 10 s, vidéo correcte) ; nb_tests ne compte que les résultats connus ; data/mesures.csv.gz mis à jour
c221198 Extraction des données en bash (pipeline/extraction.sh, curl) à la place de collect.py dans run.sh ; grille.py pour la partie .7z -> grille ; curl et unzip dans le Dockerfile
33773b7 Fusion de la branche satellite (nouvelle interface sombre) dans master
282ecd2 Suggestions d'adresses affichées devant les exemples « Essayer » (z-index du formulaire)
210b94d rasterio 1.5.1 au lieu de 1.5.2 : version précompilée disponible pour macOS 14 (Mac à puce Apple), résultats identiques
67e42d1 couverture.py : commentaires d'explication pour chaque fonction et chaque étape (code inchangé)
994bc00 clean.py : commentaires d'explication pour chaque fonction et chaque étape (code inchangé)
605c096 collect.py : commentaires d'explication pour chaque fonction et chaque étape (code inchangé)
53

---
## Étape 10 — Qualité : tests et CI/CD

📂 `tests/`, `pytest.ini`, `requirements-dev.txt`, `.github/workflows/ci.yml`

🗣️ **Quoi dire**
- **12 tests automatiques** (pytest) : distance, conversion des positions, grille (bits), lissage, poids du score, routes de l'appli. Sans Internet (l'API Adresse est remplacée par une fausse).
- **CI/CD avec GitHub Actions** : à **chaque push**, GitHub (1) lance les tests, (2) construit l'image Docker, la démarre et vérifie qu'elle répond sur `/sante`, (3) la publie sur Docker Hub (si le secret est configuré). Résultat dans l'onglet **Actions**.
- C'est le cœur du DevOps : **rien ne part en production sans vérification automatique**.

In [11]:
import sys
print(subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True, encoding="utf-8").stdout[-400:])

............                                                             [100%]
12 passed in 4.46s



---
## Étape 11 — Docker et déploiement

📂 `Dockerfile`, `.dockerignore`

🗣️ **Quoi dire**
- L'image = **Linux + Python 3.14 + librairies + projet**, lancée avec `bash run.sh` : la même chose partout (fin du « ça marche chez moi »).
- **`HEALTHCHECK`** : Docker appelle `/sante` toutes les 30 s → `docker ps` affiche `healthy` / `unhealthy`.
- Publiée sur Docker Hub : `raoufrhm/best-mobile-operator`.
```bash
docker run -p 8501:8501 -v bmo_data:/app/data raoufrhm/best-mobile-operator:latest
```
(le volume `bmo_data` garde les mises à jour du pipeline d'un lancement à l'autre)
- **Déploiement cloud** (la cible du cours) : une VM (Azure for Students, sans carte bancaire) → `docker run -d --restart unless-stopped …` → ouvrir le port 8501 → URL permanente.

In [12]:
print(open("Dockerfile", encoding="utf-8").read())

# 1) Partir d'un Linux avec Python déjà installé
FROM python:3.14-slim

# 2) Se placer dans un dossier /app à l'intérieur du conteneur
WORKDIR /app

# 3) Installer les librairies (d'abord, pour profiter du cache)
#    rasterio a besoin de libexpat1 ; l'extraction (pipeline/extraction.sh) a besoin de curl et unzip :
#    absents de l'image "slim"
RUN apt-get update && apt-get install -y --no-install-recommends libexpat1 curl unzip && rm -rf /var/lib/apt/lists/*
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 4) Copier le code du projet
COPY . .

# 5) Dire que l'appli utilise le port 8501
EXPOSE 8501

# 6) Supervision : toutes les 30 s, Docker vérifie que l'appli répond sur /sante
#    (« docker ps » affiche alors healthy ou unhealthy). Python suffit, pas besoin de curl.
HEALTHCHECK --interval=30s --timeout=5s --start-period=60s --retries=3     CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8501/sante', timeout=4)"

# 7) La co

---
## Étape 12 — Limites, et questions probables du board

🗣️ **Limites (à dire avant qu'on nous les dise)**
- Une antenne proche ≠ bonne réception à l'intérieur (murs, relief).
- Mesures Arcep dans ~1 600 communes seulement ; ailleurs, moyenne nationale (les 4 opérateurs ont alors les mêmes taux et débit).
- Couverture = **4G**, **simulée** par les opérateurs ; pas d'outre-mer.
- Le résultat est une **estimation**, pas un test avec son propre téléphone.

**Questions probables**

| Question | Réponse courte |
|---|---|
| Pourquoi ces poids ? | Les mesures réelles (50 %) pèsent plus que la simulation (30 %) et les antennes (20 %) |
| Et s'il n'y a pas de mesure près de chez moi ? | Lissage vers la moyenne nationale (`K = 20`) |
| Pourquoi les données dans Git ? | Démarrage immédiat après un clone : 27 Mo au lieu de 1,5 Go |
| Que se passe-t-il quand l'ANFR publie ? | `collect.py` le voit (carnet), télécharge ; `clean.py` nettoie ; l'appli relit toute seule |
| Comment vous savez que ça marche ? | 12 tests + CI à chaque push + `/sante` |
| Qu'est-ce qui a été difficile ? | Fichiers lourds (3 Go → grille), dossiers Arcep déplacés, fins de ligne Windows, place sur le disque |

**Ce qu'il reste à faire :** déployer sur un serveur (URL permanente), préparer le pitch, bonus éventuels (ML 5G, curseurs de poids).